# YOLOv8 Obstacle Detection — SolarSense AR

Trains YOLOv8 nano/small on rooftop obstacle images.
Classes: AC Unit, Water Tank, Vent, Chimney, Satellite Dish

> Note: Replace `dataset.yaml` path with your actual annotated dataset.
> Dataset format: COCO → YOLO conversion via Roboflow or LabelImg.


In [ ]:
# !pip install ultralytics onnx

In [ ]:
from ultralytics import YOLO
import yaml
import os

print('Ultralytics YOLO loaded ✓')

In [ ]:
# ── Dataset config (YAML) ─────────────────────────────────────────────────────
dataset_config = {
    'path': './rooftop_obstacles_dataset',   # update to actual path
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 5,
    'names': ['ac_unit', 'water_tank', 'vent', 'chimney', 'satellite_dish']
}

with open('rooftop_obstacles.yaml', 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print('Dataset config written → rooftop_obstacles.yaml')
print(f"Classes: {dataset_config['names']}")

In [ ]:
# ── Load pretrained YOLOv8n (nano — fast inference on mobile backend) ─────────
model = YOLO('yolov8n.pt')  # downloads automatically on first run
print(model.info())

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────────
# WARNING: Requires annotated dataset in rooftop_obstacles.yaml path.
# Remove this guard (os.path.exists check) when dataset is ready.

DATASET_READY = os.path.exists('./rooftop_obstacles_dataset')

if DATASET_READY:
    results = model.train(
        data='rooftop_obstacles.yaml',
        epochs=100,
        imgsz=640,
        batch=16,
        device='0',         # use '0' for GPU, 'cpu' for CPU
        name='rooftop_v1',
        patience=20,         # early stopping
        augment=True,
        mosaic=1.0,
        mixup=0.1,
        copy_paste=0.1,
    )
    print('Training complete!')
    print(f'Best mAP50: {results.results_dict["metrics/mAP50(B)"]:.3f}  (target: >0.85)')
else:
    print('Dataset not found — training skipped.')
    print('Download a rooftop dataset from Roboflow Universe:')
    print('  https://universe.roboflow.com/search?q=rooftop+objects')

In [ ]:
# ── Validation metrics ────────────────────────────────────────────────────────
if DATASET_READY:
    metrics = model.val(data='rooftop_obstacles.yaml')
    print(f'mAP50:     {metrics.box.map50:.3f}')
    print(f'mAP50-95:  {metrics.box.map:.3f}')
    print(f'Precision: {metrics.box.mp:.3f}')
    print(f'Recall:    {metrics.box.mr:.3f}')

In [ ]:
# ── Demo inference on a test image ───────────────────────────────────────────
# Replace path with an actual rooftop image
test_image_path = './test_rooftop.jpg'

if os.path.exists(test_image_path):
    results = model(test_image_path, conf=0.4)
    print(f'Detected objects: {len(results[0].boxes)}')
    for box in results[0].boxes:
        cls_id = int(box.cls)
        conf = float(box.conf)
        label = dataset_config['names'][cls_id]
        print(f'  {label}: {conf:.2f} confidence')
    results[0].show()
else:
    print('Place a test rooftop image at ./test_rooftop.jpg to run inference.')

In [ ]:
# ── Export trained model to ONNX ──────────────────────────────────────────────
if DATASET_READY:
    model.export(
        format='onnx',
        dynamic=True,
        simplify=True,
        opset=17,
        half=False  # set True for FP16 if GPU supports it
    )
    print('✓ ONNX model exported → yolov8n_rooftop.onnx')